# 04 — Filtragem baseada em conteúdo

**EP · Data Science Knowledge Base | Sistemas de Recomendação | Notebook 04/16**

**Execução:** Reinicie o kernel e execute as células em ordem. Observe as instruções específicas e as dependências opcionais indicadas neste notebook.


## Materiais complementares

- [Guia Técnico](reference_materials/04_filtragem_conteudo.pdf)
- [Cheat Sheet](cheat_sheets/cheat_sheet_04_filtragem_conteudo.pdf)

**Navegação:** os links são relativos à pasta deste notebook.

## Objetivos de aprendizagem
- Representar gêneros como vetores
- Aplicar similaridade cosseno e excluir item-semente

## Fundamentação conceitual
A similaridade cosseno de dois vetores $a,b$ é $a^Tb/(\|a\|\|b\|)$. Vetores binários de gêneros são um baseline simples; TF-IDF ou embeddings podem enriquecer descrições.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.metrics.pairwise import cosine_similarity
DATA = Path('datasets')
if not (DATA / 'ratings.csv').exists():
    raise FileNotFoundError('Abra o JupyterLab na pasta da Disciplina 09 (onde existe datasets/).')
ratings = pd.read_csv(DATA / 'ratings.csv')
movies = pd.read_csv(DATA / 'movies.csv')
print('Avaliações:', len(ratings), '| Filmes:', len(movies), '| Usuários:', ratings.userId.nunique())

Avaliações: 100836 | Filmes: 9742 | Usuários: 610


## Matriz de gêneros

In [2]:
from sklearn.preprocessing import MultiLabelBinarizer
genres=movies.genres.fillna('').str.split('|')
mlb=MultiLabelBinarizer(); X=mlb.fit_transform(genres)
print('Forma:',X.shape,'gêneros:',list(mlb.classes_)[:8])

Forma: (9742, 20) gêneros: ['(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary']


**Interpretação.** O formato binário capta sobreposição de gêneros, não semântica fina.

## Recomendações similares a Toy Story

In [3]:
pos=movies.index[movies.movieId.eq(1)][0]
sims=cosine_similarity(X[pos:pos+1],X).ravel(); sims[pos]=-1
best=np.argsort(-sims)[:10]
print(movies.iloc[best][['title','genres']].assign(sim=sims[best]).to_string(index=False))

                                                  title                                      genres  sim
         Adventures of Rocky and Bullwinkle, The (2000) Adventure|Animation|Children|Comedy|Fantasy  1.0
                                     Toy Story 2 (1999) Adventure|Animation|Children|Comedy|Fantasy  1.0
                                           Turbo (2013) Adventure|Animation|Children|Comedy|Fantasy  1.0
                                 Shrek the Third (2007) Adventure|Animation|Children|Comedy|Fantasy  1.0
                                            Antz (1998) Adventure|Animation|Children|Comedy|Fantasy  1.0
                                  Monsters, Inc. (2001) Adventure|Animation|Children|Comedy|Fantasy  1.0
                       Emperor's New Groove, The (2000) Adventure|Animation|Children|Comedy|Fantasy  1.0
                                           Moana (2016) Adventure|Animation|Children|Comedy|Fantasy  1.0
Asterix and the Vikings (Astérix et les Vikings) (2006)

**Interpretação.** Itens com mesmos gêneros recebem pontuações equivalentes; desempate merece atenção.

## Aprofundamento e aplicação

**Ponto de atenção deste capítulo:** metadados e cold start. Este complemento não substitui as seções e exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **metadados e cold start** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **metadados e cold start** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.


## Análise crítica — Perfil por conteúdo e semelhança não são personalização

A similaridade do item-semente com outros itens é um **item-to-item baseado em atributos**. Não cria um perfil pessoal. Em gêneros binários, interseções elevadas provocam empates. TF-IDF ou embeddings podem enriquecer a representação, mas exigem avaliação de cobertura, explicabilidade e adequação ao domínio.

**Origem:** aprofundamento técnico autoral fundamentado no experimento deste notebook. Não é transcrição dos materiais históricos.

In [4]:
# Inspecionar empates de similaridade no catálogo
scores = pd.Series(sims, index=movies.movieId)
validos = scores[scores.ge(0)]
print('Itens com similaridade máxima (excluído o item-semente):', int(validos.eq(validos.max()).sum()))
assert 1 not in validos.index or scores.loc[1] < 0

Itens com similaridade máxima (excluído o item-semente): 12


### Critério de leitura

Interprete os resultados como diagnóstico didático. Para comparar modelos de recomendação, defina previamente população, itens elegíveis, janela temporal, sinal de relevância, baseline e métrica; mantenha treino, validação e teste separados.

**Atividade:** registre o efeito de uma mudança de parâmetro e explique uma ameaça à validade do experimento.

## Limitações e boas práticas
Gêneros são metadados limitados. Conteúdo excessivamente semelhante pode reduzir a descoberta.

## Exercícios
Experimente outro filme e observe o efeito de um gênero muito frequente.

## Soluções comentadas
`pos=movies.index[movies.movieId.eq(50)][0]` e repita a consulta de cosseno.

## Síntese
Reveja os resultados e compare métodos usando um protocolo de dados e métricas consistente. Não trate avaliações ausentes como avaliações negativas.

### Exportação de resultados

A próxima célula grava os resultados do capítulo em `outputs/`.


In [5]:
# Exportação após os experimentos. Executar este notebook desde o início (Run All).
# Arquivos com prefixo do capítulo evitam colisões com outras saídas.
from pathlib import Path
out = Path('outputs')
out.mkdir(parents=True, exist_ok=True)
pd.DataFrame({"movieId": movies.iloc[best].movieId.to_numpy(),
              "title": movies.iloc[best].title.to_numpy(),
              "similaridade_generos": sims[best]}).to_csv(
    out / "04_similares_por_conteudo.csv", index=False)
print('Outputs do capítulo 04 salvos em:', out.resolve())


Outputs do capítulo 04 salvos em: /run/media/eduardo/Backup/Faculdades/PUC - CIENCIA DE DADOS/PUC/02_Disciplinas/09_recommender_systems/outputs
